# Practical 4: Autoencoder for anomaly detection (ECG heartbeats)

**What we are doing.** The data is 4,998 heartbeats, each 140 readings, and the last column is 1 = normal, 0 = anomaly. An **autoencoder** is trained to **copy its input to its output** through a narrow bottleneck (140 → 8 → 140). We train it **only on normal beats**, so it redraws normal beats well and abnormal ones badly. A high **reconstruction error** means an anomaly, and a **threshold** decides.

```
beat 140 → ENCODER 32 → 16 → 8 (latent code) → DECODER 16 → 32 → 140 → redrawn beat
error = mean |input − redrawn|     error > threshold → ANOMALY
```
**Skeleton:** imports → load → preprocess → **model** → compile → fit → evaluate → plot/predict

> **Run locally in Jupyter?** Install once: `%pip install tensorflow numpy pandas matplotlib scikit-learn`, then *Kernel → Restart Kernel*.
> **In Colab:** *Runtime → Change runtime type → T4 GPU*. Everything is preinstalled.

## a. Import libraries
`pandas` reads the CSV. `sklearn` splits the data and computes the metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from tensorflow import keras
from tensorflow.keras import layers

## b. Access the dataset, split, scale
- `data[:, :-1]` = the 140 readings, `data[:, -1]` = the label.
- 80/20 train/test split. `random_state=21` gives the same split every run.
- Min-max scale to 0–1 using the **training** min/max only, so the test data never influences training.
- `x_train[y_train == 1]` is a boolean mask that keeps only the normal beats.

In [ ]:
df = pd.read_csv('http://storage.googleapis.com/download.tensorflow.org/data/ecg.csv', header=None)
data = df.values
X, y = data[:, :-1], data[:, -1]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

# Scale to 0-1 using the TRAINING min and max only
mn, mx = x_train.min(), x_train.max()
x_train = (x_train - mn) / (mx - mn)
x_test = (x_test - mn) / (mx - mn)

normal_train = x_train[y_train == 1]          # train on NORMAL beats only
print(x_train.shape, x_test.shape, normal_train.shape)

## c–e. Encoder, decoder, compile
- Encoder funnel **140 → 32 → 16 → 8**. The decoder mirrors it: **8 → 16 → 32 → 140**.
- **sigmoid** output keeps the values in 0–1, like the scaled input.
- **loss = mae** (mean absolute error), because the output is a curve of numbers, not a class.

In [ ]:
# c. Encoder: 140 -> 32 -> 16 -> 8 (latent representation)
encoder = keras.Sequential([
    keras.Input(shape=(140,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(8, activation='relu'),
])
# d. Decoder: 8 -> 16 -> 32 -> 140 (back to the original size)
decoder = keras.Sequential([
    keras.Input(shape=(8,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(140, activation='sigmoid'),   # outputs in 0-1, like the scaled input
])
autoencoder = keras.Sequential([encoder, decoder])

# e. Compile with optimizer, loss and evaluation metric
autoencoder.compile(optimizer='adam', loss='mae', metrics=['mse'])

## Train: input = target
`fit(normal_train, normal_train)`: the target *is* the input. No labels are needed, which is called **self-supervised** learning.

In [ ]:
history = autoencoder.fit(normal_train, normal_train, epochs=20, batch_size=512,
                          validation_split=0.1)

plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Reconstruction loss'); plt.xlabel('Epoch'); plt.legend(); plt.show()

## Threshold and evaluate
- Reconstruction error per beat = mean |redrawn − original| over the 140 points.
- **Threshold = mean + 1 std** of the error on normal training beats.
- **Accuracy:** share labelled correctly. **Precision:** of the beats called normal, how many really were. **Recall:** of the truly normal beats, how many were caught.

In [ ]:
# Reconstruction error per beat = mean absolute difference between input and output
def recon_error(x):
    return np.mean(np.abs(autoencoder.predict(x, verbose=0) - x), axis=1)

train_err = recon_error(normal_train)
threshold = train_err.mean() + train_err.std()
print("Threshold:", threshold)

# Evaluate: error below threshold -> normal, above -> anomaly
test_err = recon_error(x_test)
pred_normal = test_err < threshold
actual_normal = y_test == 1
print("Accuracy :", accuracy_score(actual_normal, pred_normal))
print("Precision:", precision_score(actual_normal, pred_normal))
print("Recall   :", recall_score(actual_normal, pred_normal))

# Histogram: normal beats have low error, anomalies high
plt.hist(test_err[actual_normal], bins=50, alpha=0.6, label='normal')
plt.hist(test_err[~actual_normal], bins=50, alpha=0.6, label='anomaly')
plt.axvline(threshold, color='k', linestyle='--', label='threshold')
plt.xlabel('Reconstruction error'); plt.legend(); plt.show()

---
## Understand it: poke cells
**A. A normal beat vs an abnormal one**

In [ ]:
plt.plot(x_train[y_train == 1][0], label='normal'); plt.plot(x_train[y_train == 0][0], label='anomaly'); plt.legend(); plt.show()

**B. The bottleneck:** 140 numbers squeezed into 8.

In [ ]:
print(encoder(normal_train[:1]).shape)   # (1, 8)

**C. Reconstruction:** the normal beat is traced closely and the anomaly is not. The gap is the reconstruction error.

In [ ]:
for x, name in [(x_test[y_test == 1][:1], 'normal'), (x_test[y_test == 0][:1], 'anomaly')]:
    r = autoencoder.predict(x, verbose=0)
    plt.plot(x[0], label='input'); plt.plot(r[0], label='reconstruction'); plt.title(name); plt.legend(); plt.show()

---
## Break it
Train on `x_train` (**all** beats) instead of `normal_train`. The autoencoder learns to redraw anomalies too, the histograms overlap more, and accuracy drops. That's why it's trained on normal data only.

---
## Viva answers
**Q1. Why input = target, and why only normal data?** It learns to reproduce its input (self-supervised, no labels needed). If it only sees normal beats, only those are reproduced well, so anomalies stand out by their high error.

**Q2. Encoder, latent code, decoder?** The encoder compresses 140 → 8, the 8 numbers are the latent representation (bottleneck), and the decoder expands 8 → 140. The bottleneck forces it to learn the main pattern instead of just copying.

**Q3. How is an anomaly decided?** Compute the reconstruction error (mean absolute difference). If it's above the threshold (mean + std of the normal training error), the beat is an anomaly.

> **Classmate's code** uses `AirQuality.csv` with no labels, so it can't report accuracy. If the exam gives you AirQuality, the model stays the same and only the loading changes: `sep=';'`, commas → dots, −200 → NaN, fill with the mean, `StandardScaler`, and a linear output with `mse`.